In [1]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip install transformers pillow scikit-image scikit-learn
!pip install kaggle tensorboard
!pip install nltk rouge-score

Looking in indexes: https://download.pytorch.org/whl/cu130
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1087bfc20c1c71681e9d1aa21840120cb459bfbb105225e17736ca625d0bee4e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from torchvision import models, transforms

import os
import json
import numpy as np
from PIL import Image
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import math

warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
if device.type == 'cuda':
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  CUDA Version: {torch.version.cuda}")
    torch.cuda.empty_cache()
else:
    print("  ⚠️  WARNING: GPU not available! Training will be very slow.")
    print("  Go to Runtime → Change runtime type → Select GPU")

✓ Device: cuda
  GPU Name: Tesla T4
  GPU Memory: 15.83 GB
  CUDA Version: 12.6


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zemddx/vist-data")

print("Path to dataset files:", path)

100%|██████████| 3.77G/3.77G [00:43<00:00, 92.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/zemddx/vist-data/versions/2


In [9]:
from google.colab import drive
import os
import shutil
import time

# 1️⃣ Mount Google Drive
drive.mount('/content/drive')

# 2️⃣ Source: KaggleHub cache (already downloaded)
SOURCE_PATH = "/root/.cache/kagglehub/datasets/zemddx/vist-data/versions/2"

# 3️⃣ Destination: Google Drive
DEST_PATH = "/content/drive/MyDrive/vist_kaggle"

# 4️⃣ Create destination folder if it doesn't exist
os.makedirs(DEST_PATH, exist_ok=True)

print("📂 Copying dataset to Google Drive...")
print("From:", SOURCE_PATH)
print("To:  ", DEST_PATH)

start = time.time()

# 5️⃣ Copy safely (skips if already exists)
if not os.listdir(DEST_PATH):
    shutil.copytree(
        SOURCE_PATH,
        DEST_PATH,
        dirs_exist_ok=True
    )
    print("✅ Dataset copied successfully.")
else:
    print("⚠️ Dataset already exists in Drive. Skipping copy.")

end = time.time()
print(f"⏱️ Time taken: {(end - start)/60:.2f} minutes")

# 6️⃣ Verify
print("\n📁 Files in Google Drive dataset folder:")
for item in os.listdir(DEST_PATH):
    print("-", item)

print("\n✅ Dataset is now permanently stored in Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Copying dataset to Google Drive...
From: /root/.cache/kagglehub/datasets/zemddx/vist-data/versions/2
To:   /content/drive/MyDrive/vist_kaggle
⚠️ Dataset already exists in Drive. Skipping copy.
⏱️ Time taken: 0.00 minutes

📁 Files in Google Drive dataset folder:
- archive (18)

✅ Dataset is now permanently stored in Google Drive.


In [14]:
dataset_path = '/content/drive/MyDrive/VIST'

# Find JSON files
json_files = [f for f in os.listdir(dataset_path) if f.endswith('.json')]
print(f"📋 JSON files found: {json_files}\n")

# Load and inspect first JSON file
if 'train.story-in-sequence.json' in json_files:
    json_file = 'train.story-in-sequence.json'
else:
    json_file = json_files[0]

json_path = os.path.join(dataset_path, json_file)

print(f"Inspecting {json_file}...")
with open(json_path, 'r') as f:
    sample_data = json.load(f)

# Handle different data formats
if isinstance(sample_data, dict):
    first_key = list(sample_data.keys())[0]
    sample_story = sample_data[first_key]
    print(f"Data format: Dictionary")
    print(f"Number of stories: {len(sample_data)}")
elif isinstance(sample_data, list):
    sample_story = sample_data[0]
    print(f"Data format: List")
    print(f"Number of stories: {len(sample_data)}")

print(f"\n📖 Sample story structure:")
print(json.dumps(sample_story, indent=2)[:500])

print(f"\nKey fields in story:")
if isinstance(sample_story, dict):
    for key in sample_story.keys():
        if key in ['story_id', 'photo_ids', 'sentences', 'descriptions']:
            value = sample_story[key]
            if isinstance(value, list):
                print(f"  ✓ {key}: {len(value)} items")
            else:
                print(f"  ✓ {key}: {value}")

📋 JSON files found: ['test.story-in-sequence.json', 'train.story-in-sequence.json', 'val.story-in-sequence.json']

Inspecting train.story-in-sequence.json...
Data format: Dictionary
Number of stories: 5

📖 Sample story structure:
[
  {
    "datetaken": "2008-06-30 07:33:43",
    "license": "5",
    "title": "Moreton Bay Fig 1877",
    "text": "",
    "album_id": "72157605930515606",
    "longitude": "-119.692879",
    "url_o": "https://farm3.staticflickr.com/2078/2626977325_2b7696990c_o.jpg",
    "secret": "bec0ff3596",
    "media": "photo",
    "latitude": "34.414760",
    "id": "2626977325",
    "tags": "santabarbara"
  },
  {
    "datetaken": "2008-06-30 07:34:04",
    "license": "5",
    "title": "Santa Barbara",
   

Key fields in story:


In [12]:
import json
import os
from collections import Counter
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class VISTDataset(Dataset):
    """
    Visual Storytelling (VIST) Dataset
    Handles both dict and list story formats

    Fix 2 support:
    - Build vocab only once (train)
    - Reuse train vocab for val/test by passing word2idx/idx2word or calling set_vocab()
    """

    def __init__(
        self,
        dataset_path,
        json_file,
        transform=None,
        min_freq=2,
        img_size=224,
        build_vocab=True,
        word2idx=None,
        idx2word=None,
    ):
        self.dataset_path = dataset_path
        self.img_size = img_size
        self.transform = transform or transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # Load JSON file
        json_path = os.path.join(dataset_path, json_file)
        print(f"Loading: {json_path}")

        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Extract stories based on JSON structure
        if isinstance(data, dict):
            if "annotations" in data:
                self.stories = data["annotations"]
            elif "stories" in data:
                self.stories = data["stories"]
            else:
                first_key = list(data.keys())[0]
                self.stories = data[first_key] if isinstance(data[first_key], list) else [data]
        elif isinstance(data, list):
            self.stories = data
        else:
            raise ValueError("Unknown JSON format")

        print(f"Loaded {len(self.stories)} items")

        # Build photo ID → image path mapping
        self.photo_id_map = self._build_photo_id_map()
        print(f"Found {len(self.photo_id_map)} images")

        # ---- Vocabulary handling (Fix 2) ----
        # Always have special tokens
        default_word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        default_idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}

        if word2idx is not None and idx2word is not None:
            # Use externally provided vocab (recommended for val/test)
            self.word2idx = word2idx
            self.idx2word = idx2word
            print(f"Vocabulary size (shared): {len(self.word2idx)}")
        else:
            # Build or not build depending on flag
            self.word2idx = dict(default_word2idx)
            self.idx2word = dict(default_idx2word)

            if build_vocab:
                self._build_vocab(min_freq)
                print(f"Vocabulary size (built): {len(self.word2idx)}")
            else:
                # build_vocab=False but no vocab passed => still usable, but mostly <unk>
                print(f"Vocabulary size (default-only): {len(self.word2idx)}")

    def set_vocab(self, word2idx, idx2word):
        """Attach external vocabulary after dataset is created (train -> val/test)."""
        self.word2idx = word2idx
        self.idx2word = idx2word
        print(f"Vocabulary size (shared): {len(self.word2idx)}")

    def _build_photo_id_map(self):
        """Create mapping from photo_id to file path"""
        photo_id_map = {}
        for root, _, files in os.walk(self.dataset_path):
            for file in files:
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    photo_id = os.path.splitext(file)[0]
                    photo_id_map[photo_id] = os.path.join(root, file)
        return photo_id_map

    def _build_vocab(self, min_freq):
        """Build vocabulary from all stories (training only)"""
        word_freq = Counter()
        text_found_count = 0

        for story in self.stories:
            text_list = []

            if isinstance(story, list):
                for item in story:
                    if isinstance(item, dict) and "text" in item:
                        text_list.append(item["text"])
            elif isinstance(story, dict) and "sentences" in story:
                for sent in story["sentences"]:
                    if isinstance(sent, dict) and "text" in sent:
                        text_list.append(sent["text"])
            elif isinstance(story, dict):
                for key in ["text", "description", "original_text"]:
                    if key in story:
                        text_list.append(story[key])
                        break

            if text_list:
                text_found_count += 1
                for text in text_list:
                    tokens = str(text).lower().split()
                    word_freq.update(tokens)

        print(f"Found text in {text_found_count}/{len(self.stories)} stories")

        idx = 4
        for word, freq in word_freq.most_common():
            if freq >= min_freq and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def _load_image(self, photo_id):
        """Load image from photo_id"""
        if str(photo_id) in self.photo_id_map:
            try:
                img = Image.open(self.photo_id_map[str(photo_id)])
                if img.mode != "RGB":
                    img = img.convert("RGB")
                return self.transform(img)
            except Exception:
                return torch.zeros(3, self.img_size, self.img_size)
        return torch.zeros(3, self.img_size, self.img_size)

    def _get_text(self, story):
        """Extract text from story (dict or list)"""
        text_list = []

        if isinstance(story, list):
            for item in story:
                if isinstance(item, dict) and "text" in item:
                    text_list.append(item["text"])
        elif isinstance(story, dict) and "sentences" in story:
            for sent in story["sentences"]:
                if isinstance(sent, dict) and "text" in sent:
                    text_list.append(sent["text"])
        elif isinstance(story, dict):
            for key in ["text", "description", "original_text"]:
                if key in story:
                    text_list.append(story[key])
                    break

        return " ".join(text_list)

    def _get_photo_ids(self, story):
        """Extract photo IDs from story"""
        photo_ids = []

        if isinstance(story, list):
            for item in story:
                if isinstance(item, dict) and "photo_flickr_id" in item:
                    photo_ids.append(item["photo_flickr_id"])
        elif isinstance(story, dict):
            if "photo_flickr_id" in story:
                pid = story["photo_flickr_id"]
                photo_ids = pid if isinstance(pid, list) else [pid]

        return photo_ids

    def __len__(self):
        return len(self.stories)

    def __getitem__(self, idx):
        story = self.stories[idx]

        photo_ids = self._get_photo_ids(story)

        # Load 5 images
        images = []
        for i in range(5):
            if i < len(photo_ids):
                img = self._load_image(photo_ids[i])
            else:
                img = torch.zeros(3, self.img_size, self.img_size)
            images.append(img)
        images = torch.stack(images)  # (5, 3, 224, 224)

        # Tokenize text using CURRENT vocab (shared train vocab for val/test)
        text = self._get_text(story)
        tokens = [self.word2idx["<start>"]]
        for word in str(text).lower().split():
            tokens.append(self.word2idx.get(word, self.word2idx["<unk>"]))
        tokens.append(self.word2idx["<end>"])

        # Pad/truncate to 200
        max_len = 200
        if len(tokens) < max_len:
            tokens += [self.word2idx["<pad>"]] * (max_len - len(tokens))
        else:
            tokens = tokens[:max_len]

        texts = torch.tensor(tokens, dtype=torch.long)

        return {"images": images, "texts": texts}


print("✓ VISTDataset class defined (with shared-vocab support)")

✓ VISTDataset class defined (with shared-vocab support)


In [13]:
# ============================================================================
# CELL 6: DATA LOADERS (OPTIMIZED + FIX 2: SHARED VOCAB)
# ============================================================================

from torch.utils.data import DataLoader

# ✅ FIXED PATH (Google Drive)
dataset_path = "/content/drive/MyDrive/vist_kaggle"
print(f"Loading data from: {dataset_path}\n")

# ✅ Subsets for faster training
TRAIN_SAMPLES = 14000
VAL_SAMPLES   = 3000
TEST_SAMPLES  = 3000

print("🚀 OPTIMIZATION: Using smaller subsets")
print(f"  Train: {TRAIN_SAMPLES:,} samples (was 200,775)")
print(f"  Val:   {VAL_SAMPLES:,} samples (was 24,950)")
print(f"  Test:  {TEST_SAMPLES:,} samples (was 25,275)")
print("  → Expected epoch time depends on model + GPU\n")

try:
    # ---------------------------------------------------------------------
    # 1) TRAIN DATASET: build vocab ONCE here
    # ---------------------------------------------------------------------
    train_dataset = VISTDataset(
        dataset_path,
        "train.story-in-sequence.json",
        min_freq=5,
        build_vocab=True
    )

    # Apply subset (keeps training fast)
    train_dataset.stories = train_dataset.stories[:TRAIN_SAMPLES]

    # IMPORTANT: If you want vocab built ONLY from the subset (faster, but smaller vocab),
    # you must rebuild vocab after slicing. With the current class, the vocab was built
    # during __init__ on the full train JSON. (This is OK for correctness.) [file:186]

    # ---------------------------------------------------------------------
    # 2) VAL/TEST DATASETS: DO NOT build vocab. Reuse train vocab (Fix 2)
    # ---------------------------------------------------------------------
    val_dataset = VISTDataset(
        dataset_path,
        "val.story-in-sequence.json",
        build_vocab=False,
        word2idx=train_dataset.word2idx,
        idx2word=train_dataset.idx2word
    )

    test_dataset = VISTDataset(
        dataset_path,
        "test.story-in-sequence.json",
        build_vocab=False,
        word2idx=train_dataset.word2idx,
        idx2word=train_dataset.idx2word
    )

    # Apply subsets
    val_dataset.stories  = val_dataset.stories[:VAL_SAMPLES]
    test_dataset.stories = test_dataset.stories[:TEST_SAMPLES]

    print("✅ DATASETS LOADED (shared vocab):")
    print(f"  Train: {len(train_dataset):,} stories")
    print(f"  Val:   {len(val_dataset):,} stories")
    print(f"  Test:  {len(test_dataset):,} stories")
    print(f"  Shared vocab size: {len(train_dataset.word2idx):,}\n")

    # ---------------------------------------------------------------------
    # 3) DATALOADER OPTIMIZATION
    # ---------------------------------------------------------------------
    batch_size = 32
    num_workers = 4
    pin_memory = True
    prefetch_factor = 2

    print("🚀 DATALOADER SETTINGS:")
    print(f"  Batch size: {batch_size}")
    print(f"  Num workers: {num_workers}")
    print(f"  Pin memory: {pin_memory}")
    print(f"  Prefetch factor: {prefetch_factor}\n")

    # persistent_workers only works when num_workers > 0
    persistent_workers = True if num_workers > 0 else False

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        prefetch_factor=prefetch_factor,
        persistent_workers=persistent_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        prefetch_factor=prefetch_factor,
        persistent_workers=persistent_workers,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        prefetch_factor=prefetch_factor,
        persistent_workers=persistent_workers,
    )

    print("✅ DATALOADERS CREATED:")
    print(f"  Train batches/epoch: {len(train_loader)}")
    print(f"  Val batches:         {len(val_loader)}")
    print(f"  Test batches:        {len(test_loader)}\n")

    # ---------------------------------------------------------------------
    # 4) Sanity check
    # ---------------------------------------------------------------------
    print("🔍 Testing first batch...")
    batch = next(iter(train_loader))
    print(f"  Images shape: {batch['images'].shape}")  # [B, 5, 3, 224, 224]
    print(f"  Texts shape:  {batch['texts'].shape}")   # [B, 200]
    print("✅ DataLoaders ready for training! 🚀")

except Exception as e:
    print(f"\n❌ ERROR LOADING DATA: {e}")
    print("Please check if your Drive is mounted and paths are correct.")
    import traceback
    traceback.print_exc()

Loading data from: /content/drive/MyDrive/vist_kaggle

🚀 OPTIMIZATION: Using smaller subsets
  Train: 14,000 samples (was 200,775)
  Val:   3,000 samples (was 24,950)
  Test:  3,000 samples (was 25,275)
  → Expected epoch time depends on model + GPU

Loading: /content/drive/MyDrive/vist_kaggle/train.story-in-sequence.json

❌ ERROR LOADING DATA: [Errno 2] No such file or directory: '/content/drive/MyDrive/vist_kaggle/train.story-in-sequence.json'
Please check if your Drive is mounted and paths are correct.


Traceback (most recent call last):
  File "/tmp/ipython-input-3651497672.py", line 26, in <cell line: 0>
    train_dataset = VISTDataset(
                    ^^^^^^^^^^^^
  File "/tmp/ipython-input-3537899763.py", line 46, in __init__
    with open(json_path, "r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/vist_kaggle/train.story-in-sequence.json'


In [ ]:
class CrossModalAttentionFusion(nn.Module):
    """
    Bi-directional Cross-Modal Attention Fusion
    Allows text and image modalities to attend to each other
    """

    def __init__(self, embed_dim=512, num_heads=4, dropout=0.1):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # Text-to-Image attention (text queries, image keys/values)
        self.text_to_image = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )

        # Image-to-Text attention (image queries, text keys/values)
        self.image_to_text = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )

        # Feed-forward networks for refinement
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
        )

        # Layer normalization
        self.norm1_txt = nn.LayerNorm(embed_dim)
        self.norm2_txt = nn.LayerNorm(embed_dim)
        self.norm1_img = nn.LayerNorm(embed_dim)
        self.norm2_img = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, img_feat, txt_feat):
        """
        Args:
            img_feat: (batch, 5, embed_dim) - Image features
            txt_feat: (batch, 5, embed_dim) - Text features

        Returns:
            img_feat: (batch, 5, embed_dim) - Refined image features
            txt_feat: (batch, 5, embed_dim) - Refined text features
        """

        # Text attends to image (text queries, image keys/values)
        txt_attn, _ = self.text_to_image(txt_feat, img_feat, img_feat)
        txt_feat = self.norm1_txt(txt_feat + self.dropout(txt_attn))
        txt_feat = self.norm2_txt(txt_feat + self.dropout(self.ff(txt_feat)))

        # Image attends to text (image queries, text keys/values)
        img_attn, _ = self.image_to_text(img_feat, txt_feat, txt_feat)
        img_feat = self.norm1_img(img_feat + self.dropout(img_attn))
        img_feat = self.norm2_img(img_feat + self.dropout(self.ff(img_feat)))

        return img_feat, txt_feat

In [ ]:
# ============================================================================
# CELL 8: MEMORY-EFFICIENT MODEL (Fixes OOM)
# ============================================================================

import torch
import torch.nn as nn
from torchvision import models

class CrossModalStoryModel(nn.Module):
    """
    Memory-efficient story generation model
    Decodes one step at a time without storing intermediate outputs
    """
    def __init__(self, vocab_size, embed_dim=256, num_heads=4, hidden_dim=256,
                 num_layers=1, dropout=0.1, max_seq_len=200):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ============================================================
        # 1. IMAGE ENCODER: ResNet50 (Pre-trained, Frozen)
        # ============================================================
        self.image_encoder = models.resnet50(pretrained=True)
        # Freeze encoder to save memory
        for param in self.image_encoder.parameters():
            param.requires_grad = False
        self.image_encoder.fc = nn.Linear(2048, embed_dim)

        # ============================================================
        # 2. TEXT ENCODER: Unidirectional LSTM (Single Layer)
        # ============================================================
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.text_lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,  # Single layer to save memory
            batch_first=True,
            dropout=0 if num_layers == 1 else dropout,
            bidirectional=False  # Unidirectional to save memory
        )

        # ============================================================
        # 3. SIMPLIFIED FUSION (Just concatenation)
        # ============================================================
        # Average image features and concatenate with text
        self.fusion_proj = nn.Linear(embed_dim + hidden_dim, embed_dim)

        # ============================================================
        # 4. DECODER: Single-step LSTM
        # ============================================================
        self.decoder_lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0
        )

        # Output projection
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, images, texts):
        """
        Args:
            images: (batch_size, 5, 3, 224, 224)
            texts: (batch_size, seq_len)

        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        batch_size = images.shape[0]
        seq_len = texts.shape[1]

        # ============================================================
        # STAGE 1: IMAGE ENCODING (Frozen)
        # ============================================================
        with torch.no_grad():
            images_flat = images.view(-1, 3, 224, 224)
            img_features = self.image_encoder(images_flat)  # (batch*5, embed_dim)
            img_features = img_features.view(batch_size, 5, self.embed_dim)

            # Average 5 images into 1 context vector
            img_context = img_features.mean(dim=1)  # (batch, embed_dim)

        # ============================================================
        # STAGE 2: TEXT ENCODING
        # ============================================================
        text_embed = self.word_embed(texts)  # (batch, seq_len, embed_dim)
        text_lstm_out, (text_h, text_c) = self.text_lstm(text_embed)
        # text_h: (num_layers, batch, hidden_dim)

        # ============================================================
        # STAGE 3: FUSION - Create context for each position
        # ============================================================
        # Concatenate image context with each text position
        img_context_expanded = img_context.unsqueeze(1).expand(-1, seq_len, -1)
        # (batch, seq_len, embed_dim + embed_dim) → (batch, seq_len, embed_dim)
        fused = torch.cat([text_lstm_out, img_context_expanded], dim=2)
        fused = self.fusion_proj(fused)
        fused = self.dropout(fused)

        # ============================================================
        # STAGE 4: DECODING (Memory-Efficient)
        # ============================================================
        # Initialize hidden state
        hidden = (text_h, text_c)

        # Generate logits step-by-step (don't store intermediate results)
        logits_list = []

        for t in range(seq_len):
            # Decode one token at a time
            decoder_input = fused[:, t:t+1]  # (batch, 1, embed_dim)
            decoder_output, hidden = self.decoder_lstm(decoder_input, hidden)
            # decoder_output: (batch, 1, hidden_dim)

            logit = self.output_proj(decoder_output)  # (batch, 1, vocab_size)
            logits_list.append(logit)

        # Concatenate all logits
        logits = torch.cat(logits_list, dim=1)  # (batch, seq_len, vocab_size)

        return logits

print("✓ Memory-efficient CrossModalStoryModel defined")

✓ Memory-efficient CrossModalStoryModel defined


In [ ]:
import json
import os

print("="*70)
print("🔍 DATASET JSON INSPECTION")
print("="*70)

# 1. Locate the file
dataset_path = '/content/drive/MyDrive/vist_kaggle' # Or whatever your path is
json_file = 'train.story-in-sequence.json'
full_path = os.path.join(dataset_path, json_file)

print(f"Checking file: {full_path}")

if not os.path.exists(full_path):
    print(f"❌ File not found at {full_path}")
    # Try finding it
    print("Searching for json files...")
    for root, dirs, files in os.walk('.'):
        for file in files:
            if file.endswith('.json'):
                print(f"  Found: {os.path.join(root, file)}")
else:
    # 2. Load and Inspect
    try:
        with open(full_path, 'r') as f:
            data = json.load(f)

        print(f"✅ JSON loaded. Type: {type(data)}")

        if isinstance(data, dict):
            print(f"  Top level keys: {list(data.keys())}")

            # Usually VIST has 'annotations' or 'images'
            if 'annotations' in data:
                print(f"  'annotations' count: {len(data['annotations'])}")
                sample = data['annotations'][0]
                print(f"\n📋 SAMPLE ANNOTATION (First Item):")
                print(json.dumps(sample, indent=2))

            if 'images' in data:
                print(f"  'images' count: {len(data['images'])}")

        elif isinstance(data, list):
            print(f"  List length: {len(data)}")
            sample = data[0]
            print(f"\n📋 SAMPLE ITEM (First Item):")
            print(json.dumps(sample, indent=2))

    except Exception as e:
        print(f"❌ Error reading JSON: {e}")

print("="*70)

🔍 DATASET JSON INSPECTION
Checking file: /content/drive/MyDrive/vist_kaggle/train.story-in-sequence.json
✅ JSON loaded. Type: <class 'dict'>
  Top level keys: ['images', 'info', 'albums', 'type', 'annotations']
  'annotations' count: 200775

📋 SAMPLE ANNOTATION (First Item):
[
  {
    "original_text": "Our landmark tree in town was about to be destroyed and cleared for a new mall. ",
    "album_id": "72157605930515606",
    "photo_flickr_id": "2627795780",
    "setting": "first-2-pick-and-tell",
    "worker_id": "SY6QQXJCXXMNCYP",
    "story_id": "30355",
    "tier": "story-in-sequence",
    "worker_arranged_photo_order": 0,
    "text": "our landmark tree in town was about to be destroyed and cleared for a new mall .",
    "storylet_id": "151775"
  }
]
  'images' count: 167528


In [ ]:
REAL_VOCAB_SIZE = len(train_dataset.word2idx)
print(f"✅ Vocabulary Size: {REAL_VOCAB_SIZE}\n")

# Initialize model with memory-efficient architecture
model = CrossModalStoryModel(
    vocab_size=REAL_VOCAB_SIZE,
    embed_dim=256,          # Reduced from 512
    num_heads=4,
    hidden_dim=256,         # Reduced from 512
    num_layers=1,           # Reduced from 2 (saves memory)
    dropout=0.1,
    max_seq_len=200
)
model.to(device)

print(f"Model Architecture:")
print(f"  Embedding dim: 256 (reduced)")
print(f"  Hidden dim: 256 (reduced)")
print(f"  Num layers: 1 (single layer)")
print(f"  Frozen ResNet50: Yes (saves memory)\n")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  (ResNet50 is frozen)\n")

# Setup optimizer (smaller learning rate)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

# Learning rate scheduler
def lr_lambda(step):
    """Warmup + Cosine decay"""
    warmup_steps = 2
    total_steps = 10  # num_epochs
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    return max(0.0, float(total_steps - step) / float(max(1, total_steps - warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"✓ Training setup complete:")
print(f"  Optimizer: Adam (lr=3e-4, smaller)")
print(f"  Loss: CrossEntropyLoss (ignoring pad token)")
print(f"  LR Scheduler: Warmup + Cosine decay")

✅ Vocabulary Size: 9768

Model Architecture:
  Embedding dim: 256 (reduced)
  Hidden dim: 256 (reduced)
  Num layers: 1 (single layer)
  Frozen ResNet50: Yes (saves memory)

Model Parameters:
  Total: 30,227,560
  Trainable: 6,719,528
  (ResNet50 is frozen)



NameError: name 'lr' is not defined

In [ ]:
# See what's actually in your batch
batch = next(iter(train_loader))

print(f"Batch has {len(batch)} items:")
for i, item in enumerate(batch):
    if isinstance(item, torch.Tensor):
        print(f"  [{i}] Tensor: {item.shape}")
    else:
        print(f"  [{i}] {type(item).__name__}: {item}")

Batch has 2 items:
  [0] str: images
  [1] str: texts


In [ ]:
# ============================================================================
# CELL 10: TRAINING (FIX 1: NEXT-TOKEN SHIFT / TEACHER FORCING)
# ============================================================================

from tqdm import tqdm
import torch

def train_epoch(epoch, num_epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [TRAIN]")

    for batch_idx, batch in enumerate(pbar):
        # Unpack batch
        if isinstance(batch, dict):
            images = batch["images"]
            texts  = batch["texts"]
        else:
            images, texts = batch[0], batch[1]

        images = images.to(device, non_blocking=True)
        texts  = texts.to(device, non_blocking=True)

        # Reshape if 3D (safety)
        if texts.dim() == 3:
            texts = texts.reshape(texts.shape[0], -1)

        # -----------------------------
        # FIX 1: shift for next-token LM
        # inp:  <start> w1 w2 ... w(n-1)
        # tgt:  w1     w2 ... w(n-1) <end>
        # -----------------------------
        inp = texts[:, :-1]   # [B, L-1]
        tgt = texts[:,  1:]   # [B, L-1]

        # Forward pass
        logits = model(images, inp)  # expected: [B, L-1, V]

        # Debug first batch
        if batch_idx == 0 and epoch == 0:
            print(f"\n🔍 DEBUG: inp shape: {inp.shape}, tgt shape: {tgt.shape}, logits shape: {logits.shape}")

        # Flatten for CrossEntropyLoss: [B*(L-1), V] vs [B*(L-1)]
        logits_flat  = logits.reshape(-1, logits.shape[-1])
        targets_flat = tgt.reshape(-1)

        # Check match (should always match now)
        if logits_flat.shape[0] != targets_flat.shape[0]:
            print(f"\n❌ SHAPE MISMATCH!")
            print(f"  Logits:  {logits_flat.shape}")
            print(f"  Targets: {targets_flat.shape}")
            raise ValueError("Shape mismatch in shifted training.")

        # Loss + backward
        loss = criterion(logits_flat, targets_flat)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / len(train_loader)


def validate(epoch, num_epochs):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [VAL]")

        for batch in pbar:
            if isinstance(batch, dict):
                images = batch["images"]
                texts  = batch["texts"]
            else:
                images, texts = batch[0], batch[1]

            images = images.to(device, non_blocking=True)
            texts  = texts.to(device, non_blocking=True)

            if texts.dim() == 3:
                texts = texts.reshape(texts.shape[0], -1)

            # FIX 1: shift for validation too (same objective)
            inp = texts[:, :-1]
            tgt = texts[:,  1:]

            logits = model(images, inp)

            logits_flat  = logits.reshape(-1, logits.shape[-1])
            targets_flat = tgt.reshape(-1)

            loss = criterion(logits_flat, targets_flat)
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / len(val_loader)


# ============================================================================
# RUN TRAINING
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

num_epochs = 10
best_val_loss = float("inf")
patience = 3
patience_counter = 0

for epoch in range(num_epochs):
    train_loss = train_epoch(epoch, num_epochs)
    val_loss = validate(epoch, num_epochs)
    scheduler.step()

    print(f"✓ Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/best_model.pth")
        print("  → Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⚠️  Early stopping at epoch {epoch+1}")
            break

print("\n✅ TRAINING COMPLETE")


🚀 STARTING TRAINING


Epoch 1/10 [TRAIN]:   0%|          | 0/157 [00:00<?, ?it/s]


🔍 DEBUG: inp shape: torch.Size([32, 199]), tgt shape: torch.Size([32, 199]), logits shape: torch.Size([32, 199, 9768])


Epoch 1/10 [VAL]: 100%|██████████| 32/32 [00:17<00:00,  1.83it/s, loss=9.1879]


✓ Epoch 1: Train Loss=9.1880, Val Loss=9.1897
  → Best model saved!


...

✓ Epoch 8: Train Loss=4.6985, Val Loss=4.8184
  → Best model saved!


Epoch 9/10 [VAL]: 100%|██████████| 32/32 [00:16<00:00,  1.96it/s, loss=4.1781]


✓ Epoch 9: Train Loss=4.6521, Val Loss=4.7977
  → Best model saved!


Epoch 10/10 [VAL]: 100%|██████████| 32/32 [00:16<00:00,  1.93it/s, loss=4.1596]


✓ Epoch 10: Train Loss=4.6228, Val Loss=4.7891
  → Best model saved!

✅ TRAINING COMPLETE


In [ ]:
print("\n" + "="*70)
print("📊 EVALUATING ON TEST SET")
print("="*70 + "\n")

ckpt_path = "/content/best_model.pth"   # change to your actual path if local
# Example for Windows:
# ckpt_path = r"C:\Users\umama\Documents\Vishnu\best_model.pth"

# ------------------------------------------------------------
# 1) Re-create the model object (because you overwrote `model`)
# ------------------------------------------------------------
vocab_size = len(train_dataset.word2idx)  # must match training vocab
model = CrossModalStoryModel(vocab_size=vocab_size)
model.to(device)

# ------------------------------------------------------------
# 2) Load checkpoint correctly (DO NOT overwrite model)
# ------------------------------------------------------------
checkpoint = torch.load(ckpt_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
else:
    load_result = model.load_state_dict(checkpoint, strict=False)

print("Load report:")
print("  Missing keys   :", load_result.missing_keys)
print("  Unexpected keys:", load_result.unexpected_keys)

model.eval()


@torch.no_grad()
def test():
    """Evaluate on test set"""
    total_loss = 0.0
    pbar = tqdm(test_loader, desc="TEST SET EVALUATION")

    for batch in pbar:
        images = batch["images"].to(device, non_blocking=True)
        texts  = batch["texts"].to(device, non_blocking=True)

        logits = model(images, texts)

        bsz, seq_len, vocab_size_ = logits.shape
        logits_flat  = logits.view(bsz * seq_len, vocab_size_)
        targets_flat = texts.view(bsz * seq_len)

        loss = criterion(logits_flat, targets_flat)
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(test_loader)
    perplexity = float(np.exp(avg_loss))
    return avg_loss, perplexity


# Evaluate
test_loss, perplexity = test()

# Print results
print("\n" + "="*70)
print("📊 FINAL TEST RESULTS")
print("="*70)
print(f"Test Loss:        {test_loss:.4f}")
print(f"Test Perplexity:  {perplexity:.4f}")
print("="*70)

print("\n✓ Training pipeline complete!")
print(f"  Model loaded from: {ckpt_path}")
print("  Ready for inference and evaluation!")


# Save summary (change path if local Windows)
summary_path = "/content/training_summary.txt"
# Example Windows:
# summary_path = r"C:\Users\umama\Documents\Vishnu\training_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("CROSS-MODAL ATTENTION FUSION - TRAINING SUMMARY\n")
    f.write("="*70 + "\n\n")
    f.write("Dataset: VIST (Visual Storytelling)\n")
    f.write(f"Train samples: {len(train_dataset):,}\n")
    f.write(f"Val samples:   {len(val_dataset):,}\n")
    f.write(f"Test samples:  {len(test_dataset):,}\n")
    f.write(f"Vocabulary size: {len(train_dataset.word2idx):,}\n\n")
    f.write(f"Model Parameters: {total_params:,}\n")
    f.write(f"Trainable Parameters: {trainable_params:,}\n\n")
    f.write(f"Test Loss: {test_loss:.4f}\n")
    f.write(f"Test Perplexity: {perplexity:.4f}\n")
    f.write("="*70 + "\n")

print(f"\n✓ Summary saved to {summary_path}")


📊 EVALUATING ON TEST SET

Load report:
  Missing keys   : []
  Unexpected keys: []


TEST SET EVALUATION: 100%|██████████| 32/32 [00:16<00:00,  1.92it/s, loss=7.6124]


📊 FINAL TEST RESULTS
Test Loss:        7.5204
Test Perplexity:  1845.3397

✓ Training pipeline complete!
  Model loaded from: /content/best_model.pth
  Ready for inference and evaluation!

✓ Summary saved to /content/training_summary.txt


In [ ]:
model.eval()

batch = next(iter(test_loader))
images = batch["images"].to(device)   # [B, 5, 3, 224, 224]
texts  = batch["texts"].to(device)    # [B, 200]

# take one item
img1 = images[0:1]   # keep batch dim => [1, 5, 3, 224, 224]
gt  = texts[0].tolist()

In [ ]:
import torch
import torch.nn.functional as F

def decode_tokens(token_ids, idx2word, stop_ids=None):
    stop_ids = set(stop_ids or [])
    words = []
    for t in token_ids:
        if t in stop_ids:
            break
        words.append(idx2word.get(int(t), "<unk>"))
    return " ".join(words)

@torch.no_grad()
def sample_next_token(logits_1d, temperature=0.8, top_k=50):
    # Temperature scaling: divides logits by T before softmax [web:221]
    logits_1d = logits_1d / max(temperature, 1e-8)

    # Top-k: keep only k highest-logit tokens, then sample with multinomial [web:220]
    if top_k is not None and top_k > 0:
        k = min(int(top_k), logits_1d.numel())
        v, idx = torch.topk(logits_1d, k=k)
        probs = F.softmax(v, dim=-1)
        pick = torch.multinomial(probs, 1).item()
        return int(idx[pick].item())

    probs = F.softmax(logits_1d, dim=-1)
    return int(torch.multinomial(probs, 1).item())

@torch.no_grad()
def generate_story_sampled(
    model,
    images_1,
    word2idx,
    idx2word,
    max_len=60,
    temperature=0.8,
    top_k=50,
    block_ids=("<pad>", "<start>")
):
    model.eval()

    start = word2idx["<start>"]
    end   = word2idx["<end>"]

    # optional: prevent sampling these tokens
    blocked = [word2idx[t] for t in block_ids if t in word2idx]

    generated = [start]  # prefix grows each step

    for _ in range(max_len - 1):
        # Feed only the current prefix (no need to pad to 200)
        x = torch.tensor(generated, dtype=torch.long, device=images_1.device).unsqueeze(0)  # [1, L]
        logits = model(images_1, x)  # [1, L, vocab]

        # With shifted training (Fix 1): next token comes from the last position
        pos = x.size(1) - 1
        step_logits = logits[0, pos, :].clone()

        # block unwanted tokens
        for bid in blocked:
            step_logits[bid] = -1e9

        next_id = sample_next_token(step_logits, temperature=temperature, top_k=top_k)
        generated.append(next_id)

        if next_id == end:
            break

    # Remove <start>, stop at <end>
    return decode_tokens(generated[1:], idx2word, stop_ids=[end])


# Usage
pred_story = generate_story_sampled(
    model, img1,
    train_dataset.word2idx,
    train_dataset.idx2word,
    max_len=60,
    temperature=0.8,
    top_k=50
)

print("PRED:", pred_story)

PRED: i all was in the fireworks for the 4th .


In [ ]:
@torch.no_grad()
def continue_from_prefix(model, images_1, prefix_tokens, word2idx, idx2word, max_new_tokens=30):
    model.eval()
    pad = word2idx["<pad>"]
    end = word2idx["<end>"]

    generated = prefix_tokens[:]  # already includes <start> usually

    for _ in range(max_new_tokens):
        x = generated[:]
        if len(x) < 200:
            x = x + [pad] * (200 - len(x))
        else:
            x = x[:200]

        x = torch.tensor(x, dtype=torch.long, device=images_1.device).unsqueeze(0)
        logits = model(images_1, x)

        step = min(len(generated), 199)
        next_id = int(torch.argmax(logits[0, step, :]).item())
        generated.append(next_id)

        if next_id == end:
            break

    return decode_tokens(generated, idx2word, stop_ids=[end])

# Example: use first 20 tokens of ground truth as prefix
prefix = gt[:20]
pred_continued = continue_from_prefix(model, img1, prefix, train_dataset.word2idx, train_dataset.idx2word, max_new_tokens=30)
print("CONTINUED:", pred_continued)

CONTINUED: <start> the local <unk> holds a craft show each year .


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class VisualEncoder(nn.Module):
    """
    CNN-based image feature extractor using pre-trained ResNet
    """
    def __init__(self, feature_dim=512, pretrained=True):
        super(VisualEncoder, self).__init__()

        # Load pre-trained ResNet18 (you can use ResNet50 for better results)
        resnet = models.resnet18(pretrained=pretrained)

        # Remove the final classification layer
        # ResNet18 outputs 512 features before FC layer
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])

        # Optional: Add projection layer to desired dimension
        self.projection = nn.Linear(512, feature_dim)

        # Freeze early layers (optional, for faster training)
        for param in list(self.cnn.parameters())[:-10]:
            param.requires_grad = False

    def forward(self, images):
        """
        Args:
            images: (batch_size, 3, H, W) - RGB images
        Returns:
            features: (batch_size, feature_dim) - Image features
        """
        # Extract features
        features = self.cnn(images)  # (batch_size, 512, 1, 1)
        features = features.squeeze(-1).squeeze(-1)  # (batch_size, 512)

        # Project to desired dimension
        features = self.projection(features)  # (batch_size, feature_dim)

        return features


class VisualEncoderPatches(nn.Module):
    """
    Alternative: Extract patch-based features for attention
    Useful if you want spatial attention over image regions
    """
    def __init__(self, feature_dim=512, pretrained=True):
        super(VisualEncoderPatches, self).__init__()

        resnet = models.resnet18(pretrained=pretrained)

        # Remove avgpool and fc layers to keep spatial dimensions
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])

        # ResNet18 outputs (batch, 512, 7, 7) before avgpool
        self.projection = nn.Linear(512, feature_dim)

    def forward(self, images):
        """
        Args:
            images: (batch_size, 3, H, W)
        Returns:
            features: (batch_size, num_patches, feature_dim)
                     num_patches = 7*7 = 49 for ResNet18
        """
        features = self.cnn(images)  # (batch_size, 512, 7, 7)

        batch_size, channels, h, w = features.shape

        # Reshape to (batch_size, num_patches, channels)
        features = features.view(batch_size, channels, h * w)
        features = features.permute(0, 2, 1)  # (batch_size, 49, 512)

        # Project
        features = self.projection(features)  # (batch_size, 49, feature_dim)

        return features